# Setup Code

In [1]:
import os
import sys
from functools import partial
from pathlib import Path
from typing import Callable

import einops
import plotly.express as px
import plotly.graph_objects as go
import torch as t
from IPython.display import display
from ipywidgets import interact
from jaxtyping import Bool, Float
from torch import Tensor
from tqdm import tqdm

In [2]:
from arena_helpers.arena_utils import (
    render_lines_with_plotly,
    setup_widget_fig_ray,
    setup_widget_fig_triangle
)
sys.path.insert(0, './arena_helpers')
import arena_helpers.tests as tests

will make a helper function for plotting

# Rays & Segments

## Rays_1D

we'll make a function that generates some rays coming out of the origin. Which will take (0,0,0)

In [3]:
def make_rays_1d(num_pixels: int, y_limit: float) -> Tensor:
    """
    num_pixels: The number of pixels in the y dimension. Since there is one ray per pixel, this is
        also the number of rays.
    y_limit: At x=1, the rays should extend from -y_limit to +y_limit, inclusive of both endpoints.

    Returns: shape (num_pixels, num_points=2, num_dim=3) where the num_points dimension contains
        (origin, direction) and the num_dim dimension contains xyz.

    Example of make_rays_1d(9, 1.0): [
        [[0, 0, 0], [1, -1.0, 0]],
        [[0, 0, 0], [1, -0.75, 0]],
        [[0, 0, 0], [1, -0.5, 0]],
        ...
        [[0, 0, 0], [1, 0.75, 0]],
        [[0, 0, 0], [1, 1, 0]],
    ]
    """
    ## we'll use torch.linspace 
    y = t.linspace(-y_limit, y_limit, num_pixels)
    x= t.ones_like(y)
    z= t.zeros_like(y)
    ## stack columns
    result = t.stack([x,y,z], dim=1)
    ## add the origin points
    origin_points = t.zeros_like(result)
    return t.stack([origin_points, result], dim=1)


rays1d = make_rays_1d(9, 10.0)
fig = render_lines_with_plotly(rays1d)

## Ray-Object Intersection

Let's play with parameritization of the problem.

In [4]:
fig: go.FigureWidget = setup_widget_fig_ray()
display(fig)


@interact(v=(0.0, 6.0, 0.01), seed=(0, 10, 1))
def update(v=0.0, seed=0):
    t.manual_seed(seed)
    L_1, L_2 = t.rand(2, 2)
    P = lambda v: L_1 + v * (L_2 - L_1)
    x, y = zip(P(0), P(6))
    with fig.batch_update():
        fig.update_traces({"x": x, "y": y}, 0)
        fig.update_traces({"x": [L_1[0], L_2[0]], "y": [L_1[1], L_2[1]]}, 1)
        fig.update_traces({"x": [P(v)[0]], "y": [P(v)[1]]}, 2)

FigureWidget({
    'data': [{'type': 'scatter', 'uid': '536baad6-ae48-4338-a540-54f02f6fd98a', 'x': [], 'y': []},
             {'marker': {'size': 12},
              'mode': 'markers',
              'name': 'v=0',
              'type': 'scatter',
              'uid': '941c640f-0874-45ab-81cd-39d439196f4b',
              'x': [],
              'y': []},
             {'marker': {'size': 12, 'symbol': 'x'},
              'mode': 'markers',
              'name': 'v=1',
              'type': 'scatter',
              'uid': 'd88bca10-153f-4a77-88f2-cd52d8d2c1a9',
              'x': [],
              'y': []}],
    'layout': {'height': 400,
               'margin': {'b': 10, 'l': 40, 't': 60},
               'showlegend': False,
               'template': '...',
               'title': {'text': 'Ray coordinates illustration'},
               'width': 500,
               'xaxis': {'range': [-1.5, 2.5]},
               'yaxis': {'range': [-1.5, 2.5]}}
})

interactive(children=(FloatSlider(value=0.0, description='v', max=6.0, step=0.01), IntSlider(value=0, descript…

Suppose we have a line segment defined by points L1L1​ and L2L2​. Then for a given ray, we can test if the ray intersects the line segment like so:

- Supposing both the ray and line segment were infinitely long, solve for their intersection point.
- If the point exists, check whether that point is inside the line segment and the ray.


In [14]:
def intersect_ray_1d(ray: Float[Tensor, "points dims"], segment: Float[Tensor, "points dims"]) -> bool:
    """
    ray: shape (n_points=2, n_dim=3)  # O, D points
    segment: shape (n_points=2, n_dim=3)  # L_1, L_2 points

    Return True if the ray intersects the segment.
    """
    # extracting data
    l1, l2 = segment
    O, D = ray

    # only get x,y coordinates
    l1_l2 = (l1-l2)[0:2]
    D = D[0:2]
    l1_O = (l1-O)[0:2]

    mat_1 = t.stack([D, l1_l2], dim=1)
    if abs(t.linalg.det(mat_1)) > 1e-6:
        u_v = t.linalg.solve(mat_1, l1_O)
        v = u_v[1].item()
        u = u_v[0].item()
        return (v>=0 and v<=1) and (u >= 0)
    
    return False
    
tests.test_intersect_ray_1d(intersect_ray_1d)
tests.test_intersect_ray_1d_special_case(intersect_ray_1d)

All tests in `test_intersect_ray_1d` passed!
All tests in `test_intersect_ray_1d_special_case` passed!


## Batched Ray-Segment Intersection

Now we'll make use of useful tricks to make this operation faster for all rays and segments.

In [50]:
def intersect_rays_1d(
    rays: Float[Tensor, "nrays 2 3"], segments: Float[Tensor, "nsegments 2 3"]
) -> Bool[Tensor, " nrays"]:
    """
    For each ray, return True if it intersects any segment.
    """

    # only get the x, y coordinates on the segments and rays, using ellipsis
    rays = rays[..., :2]
    segments = segments[...,:2]
    # extract l1, l2, O, D
    l1 = segments[:, 0, :].unsqueeze(0) # shape is (1, nsegments, 1, 2)
    l2 = segments[:, 1, :].unsqueeze(0)
    O = rays[:, 0, :].unsqueeze(1) ## shape is (nrays, 1, 1, 2)
    D = rays[:, 1, :].unsqueeze(1)

    # l1 - O vector, we'll make use of broadcasting
    vec_l1_o = (l1-O)   # (nrays, nsegments, 2)

    # matrix of [[Dx , (l1-l2)x ],
    #             [Dy, (l1-l2)y]]
    vec_l1_l2 = l1-l2
    # creating our matrix
    mat_1 = t.stack([D.expand_as(vec_l1_o), vec_l1_l2.expand_as(vec_l1_o)], dim=-1)

    ## get the determinants & solve the singular issue
    dets = t.linalg.det(mat_1)
    is_singular = dets.abs() < 1e-8
    mat_1[is_singular] = t.eye(2)

    ## solve the matrices
    u_v = t.linalg.solve(mat_1, vec_l1_o).squeeze(-1)
    u = u_v[...,0]
    v = u_v[...,1]

    ## get the if ray intersects a segment
    intersected =  (u >= 0) & (v >= 0) & (v <= 1) & (~is_singular)
    return intersected.any(dim=1).squeeze(-1)

intersect_rays_1d(t.rand((5, 2,3)),t.rand(10,2,3))
tests.test_intersect_rays_1d(intersect_rays_1d)
tests.test_intersect_rays_1d_special_case(intersect_rays_1d)

All tests in `test_intersect_rays_1d` passed!
All tests in `test_intersect_rays_1d_special_case` passed!


## Make Rays 2D

Now we're going to make use of the z dimension and have rays emitted from the origin in both y and z dimensions.

In [53]:
def make_rays_2d(num_pixels_y: int, num_pixels_z: int, y_limit: float, z_limit: float) -> Float[Tensor, "nrays 2 3"]:
    """
    num_pixels_y: The number of pixels in the y dimension
    num_pixels_z: The number of pixels in the z dimension

    y_limit: At x=1, the rays should extend from -y_limit to +y_limit, inclusive of both.
    z_limit: At x=1, the rays should extend from -z_limit to +z_limit, inclusive of both.

    Returns: shape (num_rays=num_pixels_y * num_pixels_z, num_points=2, num_dims=3).
    """
    ## we'll use torch.linspace 
    result = t.zeros(num_pixels_y*num_pixels_z, 2, 3)

    y_values = t.linspace(-y_limit, y_limit, num_pixels_y)
    result[:, 1,1] = einops.repeat(y_values, "a -> (a n)", n= num_pixels_z)
    result[:, 1,0]= 1

    z_values = t.linspace(-z_limit, z_limit, num_pixels_z)
    result[:, 1,2]= einops.repeat(z_values, "a -> (n a)", n= num_pixels_z)
    return result


rays_2d = make_rays_2d(10, 10, 0.3, 0.3)
render_lines_with_plotly(rays_2d)

Great stuff!!

# Triangles

1. Understand how to parametrize triangles in 2D and 3D, and solve for their intersection with rays
2. Put everything together, to render your mesh as a 2D image

In [54]:
one_triangle = t.tensor([[0, 0, 0], [4, 0.5, 0], [2, 3, 0]])
A, B, C = one_triangle
x, y, z = one_triangle.T

fig: go.FigureWidget = setup_widget_fig_triangle(x, y, z)
display(fig)


@interact(u=(-0.5, 1.5, 0.01), v=(-0.5, 1.5, 0.01))
def update(u=0.0, v=0.0):
    P = A + u * (B - A) + v * (C - A)
    fig.update_traces({"x": [P[0]], "y": [P[1]]}, 2)

FigureWidget({
    'data': [{'marker': {'size': 12},
              'mode': 'markers+text',
              'text': [A, B, C],
              'textfont': {'size': 18},
              'textposition': 'middle left',
              'type': 'scatter',
              'uid': '5b74708e-7ad0-4d0e-bb49-37df7fb6072c',
              'x': {'bdata': 'AAAAAAAAgEAAAABA', 'dtype': 'f4'},
              'y': {'bdata': 'AAAAAAAAAD8AAEBA', 'dtype': 'f4'}},
             {'mode': 'lines',
              'type': 'scatter',
              'uid': '99d23fc0-58c3-4828-ba33-37ca5bfd8cf8',
              'x': [0.0, 4.0, 2.0, 0.0],
              'y': [0.0, 0.5, 3.0, 0.0]},
             {'marker': {'size': 12, 'symbol': 'x'},
              'mode': 'markers',
              'type': 'scatter',
              'uid': '146ad924-9b34-40b5-b3ca-ba727bfa74b2',
              'x': [],
              'y': []}],
    'layout': {'height': 400,
               'margin': {'b': 10, 'l': 40, 't': 60},
               'showlegend': False,
          

interactive(children=(FloatSlider(value=0.0, description='u', max=1.5, min=-0.5, step=0.01), FloatSlider(value…

## Intersection between triangle & ray



Given a ray with origin OO and direction DD, our intersection algorithm will consist of two steps:

- Finding the intersection between the line and the plane containing the triangle, by solving the equation P(u,v)=P(s)P(u,v)=P(s);
- Checking if uu and vv are within the bounds of the triangle.


In [62]:
Point = Float[Tensor, "points=3"]


def triangle_ray_intersects(A: Point, B: Point, C: Point, O: Point, D: Point) -> bool:
    """
    A: shape (3,), one vertex of the triangle
    B: shape (3,), second vertex of the triangle
    C: shape (3,), third vertex of the triangle
    O: shape (3,), origin point
    D: shape (3,), direction point

    Return True if the ray and the triangle intersect.
    """
    ## making the torch matrix & vector
    mat = t.stack([-D, (B-A), (C-A)], dim=1)
    vec = O-A
    s, u, v = t.linalg.solve(mat, vec)
    return ((s>=0) and (u + v <=1) and (u>=0) and (v>=0) )


tests.test_triangle_ray_intersects(triangle_ray_intersects)

All tests in `test_triangle_ray_intersects` passed!


## Storage Objects

Calling storage() on a Tensor returns a Python object wrapping the underlying C++ array. This array is 1D regardless of the dimensionality of the Tensor. This allows you to look inside the Tensor abstraction and see how the actual data is laid out in RAM.

In [65]:
x = t.rand((10,2,2))
x.storage() == x.storage() # a new python wrapper generate whenever we call storage

False

In [ ]:
y = t.rand((2,3))
z  = einops.repeat(y, "a b -> c a b", c=2)
print(f"Y: \n {y}\n")
print(f"Z: \n {z}\n")

Y: 
 tensor([[0.3121, 0.3752, 0.5355],
        [0.3268, 0.3200, 0.3743]])

Z: 
 tensor([[[0.3121, 0.3752, 0.5355],
         [0.3268, 0.3200, 0.3743]],

        [[0.3121, 0.3752, 0.5355],
         [0.3268, 0.3200, 0.3743]]])



do they share the same underlying C++ array? `einops.repeat` supposedly only returns a `view`

In [68]:
y.storage().data_ptr() == z.storage().data_ptr()

True

Yes they do share it! Let's edit `y`

In [69]:
y[1, 2] =5
print(f"Y: \n {y}\n")

Y: 
 tensor([[0.3121, 0.3752, 0.5355],
        [0.3268, 0.3200, 5.0000]])



Let's now look if `z` is changed?

In [70]:
print(f"Z: \n {z}\n")

Z: 
 tensor([[[0.3121, 0.3752, 0.5355],
         [0.3268, 0.3200, 5.0000]],

        [[0.3121, 0.3752, 0.5355],
         [0.3268, 0.3200, 5.0000]]])



Wow!